# Part B — DDoS Attack Type Recommendation System

**Student:** Benjamine. S (258762A)
**Batch:** 18  
**Programme:** MSc Artificial Intelligence  
**University:** University of Moratuwa  
**Module:** Big Data Analytics Mini Project

---

## Problem

A SOC analyst observing a DDoS attack needs to know which other attack
types to prepare defences for. This notebook builds a recommendation
system that answers that question — given an observed attack type on a
network service, recommend the most relevant attack types to watch for next.

## Users, Items and Ratings

The BCCC dataset has no explicit users or ratings. The recommendation
system is constructed using implicit feedback:

- **Users** — network service profiles grouped by destination port
  (RDP on 3389, HTTPS on 443, HTTP on 80, SMB on 445, SSH on 22/222)
- **Items** — 17 distinct DDoS attack types from the BCCC dataset
- **Ratings** — flow count of each attack type targeting each service,
  used as an implicit signal following the CF lecture definition

## Four Recommendation Layers

1. **Item-KNN Collaborative Filtering** — cosine similarity between
   attack types based on co-occurrence across service profiles
2. **ALS Matrix Factorization** — pyspark.ml.recommendation.ALS
   learns latent factors from the implicit interaction matrix
3. **Content-Based Similarity** — cosine similarity on feature centroids
   computed in Part A, same formula as the word similarity practical
4. **LLM Reflection** — GPT re-ranks CF candidates using

---
## Section 0 — Environment Setup

This section installs PySpark and OpenAI, imports all required libraries,
and creates the SparkSession.

Two configurations are added beyond Part A:

**`-Xss64m`** increases the JVM thread stack size from 512KB to 64MB.
ALS builds a deeply nested query plan. When the session has accumulated
query depth from earlier cells, serializing this plan overflows the default
stack. 64MB gives enough room for the recursion to complete.

**`checkpointDir`** is set so that `.checkpoint()` can be called before
ALS training. Checkpointing physically materialises the interaction matrix
to disk and breaks the accumulated query plan lineage. ALS then sees a
clean DataFrame with no plan history, preventing the StackOverflow
regardless of what ran before.

In [3]:
# ============================================================
# SECTION 0 — Environment Setup
# ============================================================

!pip install pyspark openai --quiet

import os
import subprocess

# --- Set JAVA_HOME inside the notebook kernel ---
# VS Code notebook kernels do not inherit terminal environment variables
# We find where Homebrew installed Java and set it explicitly
result = subprocess.run(
    ['brew', '--prefix', 'openjdk@17'],
    capture_output=True, text=True
)
JAVA_HOME = result.stdout.strip()
os.environ['JAVA_HOME'] = JAVA_HOME
os.environ['PATH'] = f"{JAVA_HOME}/bin:" + os.environ.get('PATH', '')

print(f"JAVA_HOME set to: {JAVA_HOME}")

# Verify Java is now found
java_check = subprocess.run(
    ['java', '-version'],
    capture_output=True, text=True
)
print(f"Java: {java_check.stderr.split(chr(10))[0]}")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import json

spark = SparkSession.builder \
    .appName("BCCC_DDoS_PartB") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir("./spark_checkpoints")

print(f"\nSpark version  : {spark.version}")
print(f"App name       : {spark.sparkContext.appName}")
print(f"Shuffle parts  : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Checkpoint dir : ./spark_checkpoints")

JAVA_HOME set to: /opt/homebrew/opt/openjdk@17
Java: openjdk version "17.0.19" 2026-04-21


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/13 03:10:04 WARN Utils: Your hostname, Benjamines-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.66 instead (on interface en0)
26/05/13 03:10:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 03:10:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



Spark version  : 4.0.0
App name       : BCCC_DDoS_PartB
Shuffle parts  : 8
Checkpoint dir : ./spark_checkpoints
